In [1]:
from embedder import Embedder
embedder = Embedder()
v = embedder.encode("test")
print(len(v))

2026-07-13 10:17:14.092779419 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


384


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(len(documents))

72


In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
import os
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [7]:
import json
from evaluation_utils import llm_structured

target_files = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}
subset = [d for d in documents if d['filename'] in target_files]

token_counts = []
all_records = []

for doc in subset:
    prompt = json.dumps({"filename": doc['filename'], "content": doc['content']})
    parsed, usage = llm_structured(openai_client, data_gen_instructions, prompt, Questions, model="gemini-2.5-flash")
    token_counts.append(usage.prompt_tokens)
    for q in parsed.questions:
        all_records.append({"question": q, "filename": doc['filename']})

print(sum(token_counts) / len(token_counts))

1450.6666666666667


In [8]:
import pandas as pd
ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")
print(len(ground_truth))
print(ground_truth[0])

360
{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}


In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [10]:
from minsearch import Index, VectorSearch

text_index = Index(text_fields=['content'], keyword_fields=['filename'])
text_index.fit(chunks)

vector_index = VectorSearch(keyword_fields=['filename'])
vector_index.fit(embedder.encode_batch([c['content'] for c in chunks]), chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    v = embedder.encode(query)
    return vector_index.search(v, num_results=num_results)

In [11]:
q = ground_truth[0]["question"]

print("text_search:", text_search(q)[0]['filename'])
print("vector_search:", vector_search(q)[0]['filename'])

text_search: 01-agentic-rag/lessons/03-rag.md
vector_search: 01-agentic-rag/lessons/01-intro.md


In [12]:
def compute_relevance(search_fn, gt):
    relevance = []
    for item in gt:
        results = search_fn(item['question'])
        relevance.append([r['filename'] == item['filename'] for r in results])
    return relevance

def hit_rate(relevance):
    return sum(any(r) for r in relevance) / len(relevance)

def mrr(relevance):
    total = 0
    for r in relevance:
        for i, hit in enumerate(r):
            if hit:
                total += 1 / (i + 1)
                break
    return total / len(relevance)

def evaluate(search_fn, gt):
    rel = compute_relevance(search_fn, gt)
    return {"hit_rate": hit_rate(rel), "mrr": mrr(rel)}

In [13]:
text_metrics = evaluate(text_search, ground_truth)
print(text_metrics)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}


In [14]:
vector_metrics = evaluate(vector_search, ground_truth)
print(vector_metrics)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}


In [15]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [16]:
for k in [1, 50, 100, 200]:
    fn = lambda q, k=k: hybrid_search(q, k=k)
    metrics = evaluate(fn, ground_truth)
    print(k, metrics)

1 {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}
50 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
100 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
200 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
